# Binance EMA Bot — Exploration

Use this notebook to fetch data, look at charts, and tune the EMA crossover before running the live bot.

Make sure you've copied `.env.example` to `.env` and added your **testnet** API keys from https://testnet.binance.vision/.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src.exchange import get_exchange, is_live
from src.data import fetch_history, save, load
from src.strategy import EmaCrossover, BUY, SELL
from src.backtest import run as run_backtest

print('Mode:', 'LIVE' if is_live() else 'TESTNET')

## 1. Fetch some history

In [ ]:
SYMBOL = 'BTC/USDT'
TIMEFRAME = '5m'
DAYS = 7

ex = get_exchange()
df = load(SYMBOL, TIMEFRAME)
if df is None:
    df = fetch_history(ex, symbol=SYMBOL, timeframe=TIMEFRAME, days=DAYS)
    save(df, SYMBOL, TIMEFRAME)
df.tail()

## 2. Compute EMA signals

In [ ]:
strat = EmaCrossover(fast=9, slow=21)
sig = strat.compute(df)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(sig.index, sig['close'], label='Close', alpha=0.5)
ax.plot(sig.index, sig['ema_fast'], label=f'EMA({strat.fast})')
ax.plot(sig.index, sig['ema_slow'], label=f'EMA({strat.slow})')

buys = sig[sig['signal'] == BUY]
sells = sig[sig['signal'] == SELL]
ax.scatter(buys.index, buys['close'], marker='^', color='green', s=80, label='BUY', zorder=5)
ax.scatter(sells.index, sells['close'], marker='v', color='red', s=80, label='SELL', zorder=5)

ax.set_title(f'{SYMBOL} {TIMEFRAME} — EMA crossover signals')
ax.legend()
plt.show()

## 3. Backtest

In [ ]:
result = run_backtest(df, fast=9, slow=21, starting_cash=10_000)
print(f'Final equity: ${result.final_equity:,.2f}')
print(f'Return:       {result.return_pct:+.2f}%')
print(f'Trades:       {result.n_trades}')
print(f'Win rate:     {result.win_rate*100:.1f}%')

buy_hold = (df['close'].iloc[-1] / df['close'].iloc[0] - 1) * 100
print(f'Buy & hold:   {buy_hold:+.2f}%')

result.equity_curve.plot(figsize=(14, 4), title='Equity curve');

## 4. Parameter sweep — try different EMA pairs

In [ ]:
rows = []
for fast in [5, 9, 12, 20]:
    for slow in [21, 50, 100]:
        if fast >= slow:
            continue
        r = run_backtest(df, fast=fast, slow=slow)
        rows.append({'fast': fast, 'slow': slow, 'return_pct': r.return_pct,
                     'trades': r.n_trades, 'win_rate': r.win_rate})
pd.DataFrame(rows).sort_values('return_pct', ascending=False)